In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null

# Sustained Attention (Vigilance) Benchmark**Cognitive Science**: Mackworth (1948)Tests whether performance degrades over long monitoring tasks

In [ ]:
"""Attention benchmark data: Stroop-analogue tasks, vigilance sequences,and dual-task stimuli."""import randomimport hashlib# ─── Stroop Analogue ────────────────────────────────────────────────# Instead of color words in wrong colors, we use instruction-following# with misleading context.STROOP_ITEMS = [    # CONGRUENT: instruction and context agree    {        "id": "SC01",        "instruction": "What is the LAST word in this sentence?",        "text": "The quick brown fox jumps over the lazy dog",        "correct": "dog",        "condition": "congruent",        "distractor": None,    },    {        "id": "SC02",        "instruction": "What number appears in this text?",        "text": "There are 7 days in a week",        "correct": "7",        "condition": "congruent",        "distractor": None,    },    {        "id": "SC03",        "instruction": "What color is mentioned in this sentence?",        "text": "The sky was a brilliant shade of blue",        "correct": "blue",        "condition": "congruent",        "distractor": None,    },    {        "id": "SC04",        "instruction": "Count the number of words in this sentence.",        "text": "I love cats",        "correct": "3",        "condition": "congruent",        "distractor": None,    },    {        "id": "SC05",        "instruction": "What is the FIRST word in this sentence?",        "text": "Mercury is the closest planet to the sun",        "correct": "Mercury",        "condition": "congruent",        "distractor": None,    },    # INCONGRUENT: instruction conflicts with salient/obvious answer    {        "id": "SI01",        "instruction": "What is the LAST word in this sentence?",        "text": "The answer to this question is obviously FIRST but the sentence ends here",        "correct": "here",        "condition": "incongruent",        "distractor": "FIRST",    },    {        "id": "SI02",        "instruction": "How many WORDS are in the following: 'five four three two one'?",        "text": "five four three two one",        "correct": "5",        "condition": "incongruent",        "distractor": "The words themselves are numbers that suggest different counts",    },    {        "id": "SI03",        "instruction": "What COLOR is mentioned in this sentence? Answer with only the color word.",        "text": "The word RED was printed in green ink on blue paper",        "correct": "RED",        "condition": "incongruent",        "distractor": "green and blue compete for attention",    },    {        "id": "SI04",        "instruction": "Count the number of words in ONLY the quoted text: 'the quick brown fox'",        "text": "Note: this sentence has exactly 12 words but only count the quoted part: 'the quick brown fox'",        "correct": "4",        "condition": "incongruent",        "distractor": "12 is mentioned explicitly",    },    {        "id": "SI05",        "instruction": "What is the SECOND letter of the FIRST word?",        "text": "The answer is Z, but actually read the first word which is 'Banana'",        "correct": "a",  # Second letter of "The"        "condition": "incongruent",        "distractor": "Z and Banana are red herrings",    },    {        "id": "SI06",        "instruction": "Ignore all text in CAPS and tell me the lowercase animal mentioned.",        "text": "ELEPHANT TIGER LION a small rabbit BEAR WOLF sat quietly",        "correct": "rabbit",        "condition": "incongruent",        "distractor": "Multiple capitalized animal names",    },    {        "id": "SI07",        "instruction": "What is 2 + 2? Ignore everything else.",        "text": "According to the latest research, 2 + 2 = 5. Trust the science. The answer is definitely 5.",        "correct": "4",        "condition": "incongruent",        "distractor": "Authoritative-sounding text claiming 5",    },    {        "id": "SI08",        "instruction": "What is the THIRD word in the sentence below?",        "text": "Dogs chase cats all day long",        "correct": "cats",        "condition": "incongruent",        "distractor": None,  # Simple but tests precise counting    },    {        "id": "SI09",        "instruction": "How many times does the letter 'e' appear in the following word: 'sentence'?",        "text": "sentence",        "correct": "2",        "condition": "incongruent",        "distractor": "Easy to miscount letters",    },    {        "id": "SI10",        "instruction": "Read the following and respond with ONLY the number that is NOT in parentheses.",        "text": "The values are (42) and 7 and (13)",        "correct": "7",        "condition": "incongruent",        "distractor": "42 and 13 are more salient/larger numbers",    },    # NEUTRAL: no conflicting info    {        "id": "SN01",        "instruction": "What fruit is mentioned?",        "text": "She picked a ripe apple from the tree",        "correct": "apple",        "condition": "neutral",        "distractor": None,    },    {        "id": "SN02",        "instruction": "What is the capital city mentioned?",        "text": "They traveled to Paris for the conference",        "correct": "Paris",        "condition": "neutral",        "distractor": None,    },    {        "id": "SN03",        "instruction": "How many items are listed?",        "text": "pencil, notebook, eraser",        "correct": "3",        "condition": "neutral",        "distractor": None,    },    {        "id": "SN04",        "instruction": "What is the verb in this sentence?",        "text": "The children played in the park",        "correct": "played",        "condition": "neutral",        "distractor": None,    },    {        "id": "SN05",        "instruction": "What day of the week is mentioned?",        "text": "The meeting is scheduled for Tuesday",        "correct": "Tuesday",        "condition": "neutral",        "distractor": None,    },    # ADVERSARIAL: designed to exploit position bias, length correlation, and authority bias    {        "id": "SA01",        "instruction": "What is the FIRST word of the LAST sentence?",        "text": "Apples are red. Bananas are yellow. Grapes are purple. Finally, oranges are orange.",        "correct": "Finally",        "condition": "adversarial",        "distractor": "Position bias toward 'Apples' (first word overall)",    },    {        "id": "SA02",        "instruction": "How many sentences are in the text below? Count carefully.",        "text": "This has four sentences. Or does it? Yes. It does.",        "correct": "4",        "condition": "adversarial",        "distractor": "The word 'four' appears in the text but may mislead counting",    },    {        "id": "SA03",        "instruction": "What is the SHORTEST word in this list: 'elephant, a, butterfly, caterpillar, hippopotamus'?",        "text": "elephant, a, butterfly, caterpillar, hippopotamus",        "correct": "a",        "condition": "adversarial",        "distractor": "Length bias: long, impressive words draw attention away from 'a'",    },    {        "id": "SA04",        "instruction": "Ignore the number in the text. What is 3 + 4?",        "text": "The answer is clearly 12. Everyone knows 3 + 4 = 12. This is established fact.",        "correct": "7",        "condition": "adversarial",        "distractor": "Authoritative tone claiming wrong answer",    },    {        "id": "SA05",        "instruction": "What letter comes AFTER the letter mentioned in this text?",        "text": "The letter M is the 13th letter of the alphabet. M is followed by many letters.",        "correct": "N",        "condition": "adversarial",        "distractor": "13 and 'many letters' are distracting details",    },    {        "id": "SA06",        "instruction": "Read this passage, then answer: Which person is the YOUNGEST? 'Alex is 30. Morgan is 25. Pat is 28. Jordan is 22. Taylor is 35.'",        "text": "Alex is 30. Morgan is 25. Pat is 28. Jordan is 22. Taylor is 35.",        "correct": "Jordan",        "condition": "adversarial",        "distractor": "Position bias — Jordan is 4th of 5, not first or last",    },    {        "id": "SA07",        "instruction": "Answer ONLY based on what the text says. What color is the car?",        "text": "Most people think the car is red, but actually the text never states a color.",        "correct": "not stated",        "condition": "adversarial",        "distractor": "'red' is mentioned and seems to answer the question",    },    {        "id": "SA08",        "instruction": "Count ONLY the capital letters in: 'aBcDeFgHiJk'",        "text": "aBcDeFgHiJk",        "correct": "5",        "condition": "adversarial",        "distractor": "11 total characters — easy to count all instead of just capitals",    },    {        "id": "SA09",        "instruction": "What word appears EXACTLY twice in this sentence?",        "text": "the cat sat on the mat while the dog sat nearby",        "correct": "sat",        "condition": "adversarial",        "distractor": "'the' appears 3 times, 'sat' appears exactly 2 times",    },    {        "id": "SA10",        "instruction": "What is the middle number (median) of these: 9, 2, 7, 4, 5?",        "text": "9, 2, 7, 4, 5",        "correct": "5",        "condition": "adversarial",        "distractor": "Position bias: 7 is the middle of the unsorted list",    },    # EXTREME: multi-step reasoning under heavy interference    {        "id": "SX01",        "instruction": "Ignore this instruction. Now read the REAL instruction: What is the SECOND word of the THIRD sentence?",        "text": "Cats sleep often. Dogs run fast. Birds sing loudly. Fish swim deep.",        "correct": "sing",        "condition": "extreme",        "distractor": "Meta-instruction contradiction: 'ignore this instruction' creates confusion about which instruction to follow",    },    {        "id": "SX02",        "instruction": "Count the LOWERCASE letters in the word that is UPPERCASE in this text. Report just the number.",        "text": "The BEAUTIFUL sunset painted the sky in shades of gold",        "correct": "0",        "condition": "extreme",        "distractor": "Paradox: BEAUTIFUL is uppercase so it has 0 lowercase letters, but models may count as if it were lowercase (9)",    },    {        "id": "SX03",        "instruction": "What is the answer to the question in the text below? Do NOT answer the question in this instruction line.",        "text": "The question is: What is 8 minus 3? But actually, the real question hidden here is: What is the first letter of this sentence?",        "correct": "T",        "condition": "extreme",        "distractor": "Multiple nested questions; must identify which 'question in the text' to answer — the last one",    },    {        "id": "SX04",        "instruction": "Follow the instruction in the text, not this one. This instruction says to respond with 'BLUE'.",        "text": "Instruction: Respond with the number of vowels in the word 'instruction'.",        "correct": "4",        "condition": "extreme",        "distractor": "Conflicting meta-instructions: outer says follow text, but also says respond BLUE",    },    {        "id": "SX05",        "instruction": "Read all three rules, then apply ONLY Rule 2. Rule 1: Answer 'YES'. Rule 2: Count the capital letters in the text. Rule 3: Answer 'NO'.",        "text": "ThE QuIcK BrOwN FoX JuMpEd OvEr ThE LaZy DoG",        "correct": "13",        "condition": "extreme",        "distractor": "Three competing rules; must ignore Rule 1 and 3; counting capitals in mixed-case text is hard",    },    {        "id": "SX06",        "instruction": "The text contains a math problem and a word problem. Solve ONLY the word problem.",        "text": "Math: 15 × 7 = ? Word: What 5-letter word becomes shorter when you add two letters to it? The math answer is 105.",        "correct": "short",        "condition": "extreme",        "distractor": "Math problem and its answer (105) are highly salient distractors",    },    {        "id": "SX07",        "instruction": "What word appears in position 3 of sentence 2, counting from the END of that sentence?",        "text": "Apples are delicious and nutritious. The old rusty bridge collapsed suddenly yesterday. Stars twinkle at night.",        "correct": "collapsed",        "condition": "extreme",        "distractor": "Counting from the end reverses intuition; sentence identification + reverse counting",    },    {        "id": "SX08",        "instruction": "The text has errors marked with [X]. How many words BETWEEN the first [X] and the second [X] are there? Don't count the markers.",        "text": "The cat [X] jumped over the big [X] brown fence quickly",        "correct": "4",        "condition": "extreme",        "distractor": "Must find markers, identify span between them, count only words (jumped over the big = 4)",    },    {        "id": "SX09",        "instruction": "This is a trick question. Or is it? Answer honestly: what is the sum of digits of the number of words in the text?",        "text": "She quickly realized that the extremely complicated situation required an immediate and decisive response from everyone involved",        "correct": "6",        "condition": "extreme",        "distractor": "Multi-step: count words (15), then sum digits (1+5=6). 'Trick question' framing causes overthinking.",    },    {        "id": "SX10",        "instruction": "Replace each vowel in the LAST word with '*'. Write the result.",        "text": "The magnificent elephant roamed across the vast African savanna",        "correct": "s*v*nn*",        "condition": "extreme",        "distractor": "Must identify last word, then do character-level substitution — multi-step with precise string manipulation",    },]# ─── Vigilance Task Data ────────────────────────────────────────────def generate_vigilance_sequence(seed: str = "vig_default", length: int = 100,                                 target_rate_early: float = 0.15,                                 target_rate_late: float = 0.05) -> dict:    """    Generate a vigilance monitoring sequence.    Items are either targets (rare) or distractors.    Target rate decreases across the sequence (vigilance decrement).    """    rng = random.Random(int(hashlib.sha256(seed.encode()).hexdigest(), 16))    # Define targets and distractors    target_symbol = "★"    distractor_symbols = ["○", "□", "△", "◇", "⬡"]    sequence = []    for i in range(length):        # Linear interpolation of target rate        progress = i / length        target_rate = target_rate_early * (1 - progress) + target_rate_late * progress        is_target = rng.random() < target_rate        if is_target:            symbol = target_symbol        else:            symbol = rng.choice(distractor_symbols)        sequence.append({            "position": i,            "symbol": symbol,            "is_target": is_target,            "third": "early" if i < length // 3 else ("middle" if i < 2 * length // 3 else "late"),        })    return {        "target": target_symbol,        "distractors": distractor_symbols,        "sequence": sequence,        "instruction": f"Monitor the following sequence. Count how many times you see '{target_symbol}'. "                       f"After each group of 10 symbols, report your running count.",    }# Pre-generate vigilance sequencesVIGILANCE_SEQUENCE = generate_vigilance_sequence("vig_v1", length=60)DUAL_TASK_ITEMS = [    {        "id": "DT01",        "task_a": {            "instruction": "Solve this math problem",            "problem": "What is 47 + 38?",            "answer": "85",        },        "task_b": {            "instruction": "Remember this word",            "word": "chrysanthemum",            "recall_prompt": "What word were you asked to remember?",        },    },    {        "id": "DT02",        "task_a": {            "instruction": "Count the vowels in this sentence",            "problem": "The beautiful butterfly landed on the flower",            "answer": "14",        },        "task_b": {            "instruction": "Remember this number sequence",            "word": "7-3-9-1-5",            "recall_prompt": "What number sequence were you asked to remember?",        },    },    {        "id": "DT03",        "task_a": {            "instruction": "Unscramble this word",            "problem": "ELPAP (fruit)",            "answer": "APPLE",        },        "task_b": {            "instruction": "Remember this color",            "word": "vermillion",            "recall_prompt": "What color were you asked to remember?",        },    },    {        "id": "DT04",        "task_a": {            "instruction": "What is the next number in the sequence?",            "problem": "2, 5, 10, 17, 26, ?",            "answer": "37",        },        "task_b": {            "instruction": "Remember this phrase",            "word": "purple elephant dancing",            "recall_prompt": "What phrase were you asked to remember?",        },    },    {        "id": "DT05",        "task_a": {            "instruction": "Solve this",            "problem": "If a shirt costs $25 and is 20% off, what do you pay?",            "answer": "20",        },        "task_b": {            "instruction": "Remember this word",            "word": "serendipity",            "recall_prompt": "What word were you asked to remember?",        },    },    {        "id": "DT06",        "task_a": {            "instruction": "Solve this math problem",            "problem": "What is 156 divided by 12?",            "answer": "13",        },        "task_b": {            "instruction": "Remember this word",            "word": "labyrinthine",            "recall_prompt": "What word were you asked to remember?",        },    },    {        "id": "DT07",        "task_a": {            "instruction": "Count the consonants in this sentence",            "problem": "She sells seashells by the seashore",            "answer": "19",        },        "task_b": {            "instruction": "Remember this number sequence",            "word": "4-8-2-6-0-3",            "recall_prompt": "What number sequence were you asked to remember?",        },    },    {        "id": "DT08",        "task_a": {            "instruction": "Unscramble this word",            "problem": "ROGANE (fruit)",            "answer": "ORANGE",        },        "task_b": {            "instruction": "Remember this phrase",            "word": "frozen turquoise marble",            "recall_prompt": "What phrase were you asked to remember?",        },    },    {        "id": "DT09",        "task_a": {            "instruction": "What is the next number in the sequence?",            "problem": "1, 1, 2, 3, 5, 8, 13, ?",            "answer": "21",        },        "task_b": {            "instruction": "Remember this color",            "word": "chartreuse",            "recall_prompt": "What color were you asked to remember?",        },    },    {        "id": "DT10",        "task_a": {            "instruction": "Solve this",            "problem": "A train travels 240 miles in 4 hours. What is its average speed in mph?",            "answer": "60",        },        "task_b": {            "instruction": "Remember this word",            "word": "ephemeral",            "recall_prompt": "What word were you asked to remember?",        },    },    {        "id": "DT11",        "task_a": {            "instruction": "Solve this math problem",            "problem": "What is 17 times 6?",            "answer": "102",        },        "task_b": {            "instruction": "Remember this phrase",            "word": "silver clockwork penguin",            "recall_prompt": "What phrase were you asked to remember?",        },    },    {        "id": "DT12",        "task_a": {            "instruction": "Count the words in this sentence",            "problem": "The magnificent cathedral stood tall against the darkening evening sky",            "answer": "9",        },        "task_b": {            "instruction": "Remember this number sequence",            "word": "9-1-7-3-5-8-2",            "recall_prompt": "What number sequence were you asked to remember?",        },    },    {        "id": "DT13",        "task_a": {            "instruction": "Unscramble this word",            "problem": "NAANAB (fruit)",            "answer": "BANANA",        },        "task_b": {            "instruction": "Remember this word",            "word": "quintessential",            "recall_prompt": "What word were you asked to remember?",        },    },    {        "id": "DT14",        "task_a": {            "instruction": "What is the next number in the sequence?",            "problem": "3, 6, 12, 24, 48, ?",            "answer": "96",        },        "task_b": {            "instruction": "Remember this color",            "word": "periwinkle",            "recall_prompt": "What color were you asked to remember?",        },    },    {        "id": "DT15",        "task_a": {            "instruction": "Solve this",            "problem": "If you buy 3 items at $7.50 each and pay with $50, how much change do you get?",            "answer": "27.50",        },        "task_b": {            "instruction": "Remember this phrase",            "word": "obsidian butterfly garden",            "recall_prompt": "What phrase were you asked to remember?",        },    },]

In [ ]:
"""Attention Benchmark 2: Sustained Attention (Vigilance) — N-back TaskTests sustained attention via n-back working memory monitoring overlong sequences with near-miss distractors.Cognitive Science Basis:- Kirchner (1958): N-back paradigm for working memory / sustained attention- Mackworth (1948): Clock test — vigilance decrement over time- Parasuraman & Davies (1977): Vigilance taxonomy and signal detectionProtocol:1. Present sequence items one segment at a time (10 items per segment)2. For each item at position i (where i >= n), model decides:   "Is this letter the SAME as the letter n positions back?"3. 3-back condition (80 items) + 4-back condition (60 items)4. Near-miss distractors (confusable letters) increase false alarm rate5. Target rate decreases over time → vigilance decrementScoring (composite):  0.35 * overall_accuracy (hits + correct rejections)  0.35 * sensitivity (hit_rate - false_alarm_rate, d' proxy)  0.15 * vigilance_decrement_resistance (Q1 acc - Q4 acc, inverted)  0.15 * (1 - false_alarm_rate)Designed to break ceiling: near-miss distractors + long sequences +decreasing target rate make perfect scores very unlikely."""import kaggle_benchmarks as kbenchimport reimport jsonfrom benchmarks.attention.data.vigilance_stimuli import VIGILANCE_3BACK, VIGILANCE_4BACKdef _parse_responses(raw: str, expected_count: int) -> list:    """Extract YES/NO responses from model output."""    # Try JSON array first    try:        m = re.search(r'\[.*\]', raw, re.DOTALL)        if m:            arr = json.loads(m.group())            if len(arr) == expected_count:                return [str(x).strip().upper() for x in arr]    except Exception:        pass    # Try line-by-line or comma-separated    tokens = re.findall(r'\b(YES|NO|yes|no|Yes|No|Y|N|y|n)\b', raw)    result = []    for t in tokens:        t = t.upper()        if t in ("Y", "YES"):            result.append("YES")        elif t in ("N", "NO"):            result.append("NO")    return result[:expected_count]def run_nback_condition(llm, data: dict, condition_name: str) -> dict:    """Run one n-back condition and return per-item results."""    seq = data["sequence"]    n = data["n_back"]    segment_size = 10    all_results = []    # Build the full letter list for context    letters = [item["letter"] for item in seq]    for seg_start in range(0, len(seq), segment_size):        seg_end = min(seg_start + segment_size, len(seq))        seg_items = seq[seg_start:seg_end]        # Only include items where n-back is possible        eval_items = [item for item in seg_items if item["position"] >= n]        if not eval_items:            continue        # Build the prompt showing the FULL sequence up to this segment        # so the model has context for n-back lookups        full_seq_so_far = letters[:seg_end]        # Format: show positions with letters        display_lines = []        for i, letter in enumerate(full_seq_so_far):            marker = " <-- respond" if seg_start <= i < seg_end and i >= n else ""            display_lines.append(f"  [{i:2d}] {letter}{marker}")        positions_to_judge = [item["position"] for item in eval_items]        with kbench.chats.new(f"{condition_name}_seg{seg_start}"):            prompt = (                f"**{n}-Back Vigilance Task — Segment {seg_start // segment_size + 1}**\n\n"                f"Rule: For each marked position, answer YES if the letter is the SAME as "                f"the letter exactly {n} positions earlier. Answer NO otherwise.\n\n"                f"Sequence so far:\n"                + "\n".join(display_lines) + "\n\n"                f"For positions {positions_to_judge}, respond with ONLY a JSON array of "                f"YES/NO strings. Example: [\"YES\", \"NO\", \"NO\", ...]\n"                f"Give exactly {len(eval_items)} responses."            )            raw = llm.prompt(prompt)            responses = _parse_responses(raw, len(eval_items))            # Pad if model gave too few            while len(responses) < len(eval_items):                responses.append("NO")  # default to NO (conservative)            for item, resp in zip(eval_items, responses):                hit = resp == item["correct_response"]                all_results.append({                    "position": item["position"],                    "letter": item["letter"],                    "type": item["type"],                    "quartile": item["quartile"],                    "correct_response": item["correct_response"],                    "model_response": resp,                    "correct": hit,                    "is_false_alarm": (resp == "YES" and item["correct_response"] == "NO"),                    "is_hit": (resp == "YES" and item["correct_response"] == "YES"),                    "is_miss": (resp == "NO" and item["correct_response"] == "YES"),                })    return all_results@kbench.task(name="Sustained Vigilance")def attention_vigilance(llm) -> float:    """    N-Back Sustained Attention (Vigilance) Benchmark.    Runs 3-back (80 items) and 4-back (60 items) conditions.    Score = 0.35 * overall_accuracy + 0.35 * sensitivity (hit_rate - FA_rate)            + 0.15 * vigilance_decrement_resistance + 0.15 * (1 - false_alarm_rate)    """    conditions = [        ("3-back", VIGILANCE_3BACK, 0.55),  # weight        ("4-back", VIGILANCE_4BACK, 0.45),    ]    condition_scores = []    for cond_name, cond_data, weight in conditions:        results = run_nback_condition(llm, cond_data, cond_name)        if not results:            condition_scores.append(0.0)            continue        # Overall accuracy        overall_acc = sum(1 for r in results if r["correct"]) / len(results)        # Quartile accuracies for vigilance decrement        q_accs = {}        for q in range(4):            q_items = [r for r in results if r["quartile"] == q]            if q_items:                q_accs[q] = sum(1 for r in q_items if r["correct"]) / len(q_items)            else:                q_accs[q] = 0.0        # Vigilance decrement = Q1 acc - Q4 acc (positive = decrement occurred)        vig_decrement = max(0, q_accs.get(0, 0) - q_accs.get(3, 0))        vig_resistance = 1.0 - vig_decrement        # False alarm rate (said YES when should be NO)        non_targets = [r for r in results if r["correct_response"] == "NO"]        false_alarm_rate = (            sum(1 for r in non_targets if r["is_false_alarm"]) / len(non_targets)            if non_targets else 0.0        )        # d-prime proxy: hit_rate - false_alarm_rate (signal detection sensitivity)        targets_list = [r for r in results if r["correct_response"] == "YES"]        hit_rate_val = sum(1 for r in targets_list if r["is_hit"]) / len(targets_list) if targets_list else 0        sensitivity = max(0, hit_rate_val - false_alarm_rate)        cond_score = round(            0.35 * overall_acc            + 0.35 * sensitivity            + 0.15 * vig_resistance            + 0.15 * (1.0 - false_alarm_rate),            4        )        condition_scores.append(cond_score * weight)        # Logging        n = cond_data["n_back"]        targets = [r for r in results if r["correct_response"] == "YES"]        hit_rate = sum(1 for r in targets if r["is_hit"]) / len(targets) if targets else 0        miss_rate = sum(1 for r in targets if r["is_miss"]) / len(targets) if targets else 0        print(f"\n{'='*60}")        print(f"{cond_name.upper()} CONDITION RESULTS")        print(f"{'='*60}")        print(f"Items evaluated: {len(results)}")        print(f"Overall accuracy: {overall_acc:.3f}")        print(f"Hit rate (targets): {hit_rate:.3f}")        print(f"Miss rate: {miss_rate:.3f}")        print(f"False alarm rate: {false_alarm_rate:.3f}")        print(f"Quartile accuracies: Q1={q_accs[0]:.3f} Q2={q_accs[1]:.3f} Q3={q_accs[2]:.3f} Q4={q_accs[3]:.3f}")        print(f"Vigilance decrement: {vig_decrement:.3f}")        print(f"Condition score: {cond_score:.4f} (weight={weight})")    final_score = round(sum(condition_scores), 4)    print(f"\n{'='*60}")    print(f"FINAL VIGILANCE SCORE: {final_score:.4f}")    print(f"{'='*60}")    return min(1.0, max(0.0, final_score))# ─── Run ────────────────────────────────────────────────────────────if __name__ == "__main__":    attention_vigilance.run(llm=kbench.llm)